In [1]:
import pandas as pd

In [2]:
movies = pd.read_csv("../data/raw/movies.csv")
tags = pd.read_csv("../data/raw/tags.csv")

In [3]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [5]:
movies.shape

(9742, 3)

In [6]:
tags.shape

(3683, 4)

In [7]:
tags.groupby("movieId")

In [8]:
tags.groupby("movieId").count()

,userId,tag,timestamp
movieId,,,
1,3,3,3
2,4,4,4
3,2,2,2
5,2,2,2
7,1,1,1
...,...,...,...
183611,3,3,3
184471,3,3,3
187593,3,3,3


In [9]:
tag_data = (
    tags.groupby("movieId")["tag"]
        .apply(lambda x: " ".join(x))
        .reset_index()
)

In [10]:
tag_data.head()

,movieId,tag
0,1,pixar pixar fun
1,2,fantasy magic board game Robin Williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake


In [11]:
movie_data = movies.merge(
    tag_data,
    on="movieId",
    how="left"
)

In [12]:
movie_data.head()

,movieId,title,genres,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game Robin Williams game
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,NaN
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake


In [13]:
movie_data.isnull().sum()

movieId       0
title         0
genres        0
tag        8170
dtype: int64

In [14]:
movie_data["tag"] = movie_data["tag"].fillna("")
movie_data.isnull().sum()

movieId    0
title      0
genres     0
tag        0
dtype: int64

In [15]:
movie_data["genres"] = movie_data["genres"].str.replace("|", " ", regex=False)

In [16]:
movie_data["features"] = movie_data["genres"] + " " + movie_data["tag"]

In [17]:
movie_data[["title", "features"]].head(10)

,title,features
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy pi...
1,Jumanji (1995),Adventure Children Fantasy fantasy magic board...
2,Grumpier Old Men (1995),Comedy Romance moldy old
3,Waiting to Exhale (1995),Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy pregnancy remake
5,Heat (1995),Action Crime Thriller
6,Sabrina (1995),Comedy Romance remake
7,Tom and Huck (1995),Adventure Children
8,Sudden Death (1995),Action
9,GoldenEye (1995),Action Adventure Thriller


In [18]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(stop_words="english")

In [19]:
feature_matrix = vectorizer.fit_transform(movie_data["features"])

In [20]:
feature_matrix.shape

(9742, 1677)

In [21]:
vectorizer.get_feature_names_out()[:20]

array(['06', '1900s', '1920s', '1950s', '1960s', '1970s', '1980s',
       '1990s', '2001', '250', '2d', '70mm', '80', 'aardman', 'abortion',
       'absorbing', 'abstract', 'abuse', 'academy', 'accident'],
      dtype=object)

In [22]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(feature_matrix)
similarity_matrix.shape

(9742, 9742)

In [23]:
similarity_matrix[0]

array([1.        , 0.3380617 , 0.15811388, ..., 0.        , 0.2236068 ,
       0.31622777], shape=(9742,))

In [30]:
def recommend(movie_title):
    # Find movies containing the entered text
    matches = movie_data[
        movie_data["title"].str.contains(movie_title, case=False, na=False)
    ]

    # If no movie is found
    if matches.empty:
        print(f"Movie '{movie_title}' not found.")
        return

    # If multiple movies match
    if len(matches) > 1:
        print("Multiple matches found:")
        print(matches["title"].to_list())
        print("\nUsing the first match.\n")

    # Get the first matching movie
    index = matches.index[0]
    selected_movie = movie_data.iloc[index]["title"]

    # Get similarity scores
    similar_movies = sorted(
        list(enumerate(similarity_matrix[index])),
        key=lambda x: x[1],
        reverse=True
    )

    print(f"Recommendations for '{selected_movie}':\n")

    for movie in similar_movies[1:11]:
        print(movie_data.iloc[movie[0]]["title"])

In [31]:
recommend("dark knight")
recommend("toy story")
recommend("finding nemo")

Multiple matches found:
['Dark Knight, The (2008)', 'Dark Knight Rises, The (2012)', 'Batman: The Dark Knight Returns, Part 1 (2012)', 'Batman: The Dark Knight Returns, Part 2 (2013)']

Using the first match.

Recommendations for 'Dark Knight, The (2008)':

Need for Speed (2014)
Fast Five (Fast and the Furious 5, The) (2011)
Dead Presidents (1995)
Bad Company (1995)
Faster Pussycat! Kill! Kill! (1965)
Menace II Society (1993)
Substitute, The (1996)
Nothing to Lose (1994)
Batman Returns (1992)
Monument Ave. (1998)
Multiple matches found:
['Toy Story (1995)', 'Toy Story 2 (1999)', 'Toy Story 3 (2010)']

Using the first match.

Recommendations for 'Toy Story (1995)':

Bug's Life, A (1998)
Toy Story 2 (1999)
Antz (1998)
Adventures of Rocky and Bullwinkle, The (2000)
Emperor's New Groove, The (2000)
Monsters, Inc. (2001)
Wild, The (2006)
Shrek the Third (2007)
Tale of Despereaux, The (2008)
Asterix and the Vikings (Astérix et les Vikings) (2006)
Recommendations for 'Finding Nemo (2003)':

A